In [27]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 55.7 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: Pandas
    Found existing installation: pandas 2.2.0
    Uninstalling pandas-2.2.0:
      Successfully uninstalled pandas-2.2.0


In [33]:
import pandas as pd
from PIL import Image
import numpy as np
import io
import os
import sys

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model
from pyspark.sql.functions import col, pandas_udf, PandasUDFType, element_at, split
from pyspark.sql import SparkSession

Définition des PATH pour charger les images <br /> et enregistrer les résultats

In [4]:
PATH = os.getcwd()
PATH_Data = PATH+'/data/Test1'
PATH_Result = PATH+'/data/Results'
print('PATH:        '+\
      PATH+'\nPATH_Data:   '+\
      PATH_Data+'\nPATH_Result: '+PATH_Result)

PATH:        /Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire
PATH_Data:   /Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Test1
PATH_Result: /Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Results


In [ ]:
spark = (
    SparkSession.builder
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.pyspark.python", sys.executable)
    .getOrCreate()
)

In [34]:
spark = (SparkSession
             .builder
             .appName('P8')
             .master('local')
             .config("spark.pyspark.driver.python", sys.executable)
             .config("spark.pyspark.python", sys.executable)
             .config("spark.sql.parquet.writeLegacyFormat", 'true')
             .getOrCreate()
)

26/02/06 00:13:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [35]:
sc = spark.sparkContext

In [36]:
spark

In [37]:
images = spark.read.format("binaryFile") \
  .option("pathGlobFilter", "*.jpg") \
  .option("recursiveFileLookup", "true") \
  .load(PATH_Data)

In [38]:
images = images.withColumn('label', element_at(split(images['path'], '/'),-2))
print(images.printSchema())
print(images.select('path','label').show(5,False))

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)
 |-- label: string (nullable = true)

None
+-------------------------------------------------------------------------------------------------+----------+
|path                                                                                             |label     |
+-------------------------------------------------------------------------------------------------+----------+
|file:/Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Test1/Watermelon/r_106_100.jpg|Watermelon|
|file:/Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Test1/Watermelon/r_109_100.jpg|Watermelon|
|file:/Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Test1/Watermelon/r_108_100.jpg|Watermelon|
|file:/Volumes/Work/Projets/Linda/OC/OC_P8/P8_Mode_opératoire/data/Test1/Watermelon/r_107_100.jpg|Watermelon|
|file:/Volume

In [39]:
model = MobileNetV2(weights='imagenet',
                    include_top=True,
                    input_shape=(224, 224, 3))

In [40]:
new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)

In [41]:
new_model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_2[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 112, 112, 32)         0         ['bn_Conv1[0][0]']      

In [42]:
brodcast_weights = sc.broadcast(new_model.get_weights())

In [43]:
def model_fn():
    """
    Returns a MobileNetV2 model with top layer removed 
    and broadcasted pretrained weights.
    """
    model = MobileNetV2(weights='imagenet',
                        include_top=True,
                        input_shape=(224, 224, 3))
    for layer in model.layers:
        layer.trainable = False
    new_model = Model(inputs=model.input,
                  outputs=model.layers[-2].output)
    new_model.set_weights(brodcast_weights.value)
    return new_model

In [45]:
def preprocess(content):
    """
    Preprocesses raw image bytes for prediction.
    """
    img = Image.open(io.BytesIO(content)).resize([224, 224])
    arr = img_to_array(img)
    return preprocess_input(arr)

def featurize_series(model, content_series):
    """
    Featurize a pd.Series of raw images using the input model.
    :return: a pd.Series of image features
    """
    input = np.stack(content_series.map(preprocess))
    preds = model.predict(input)
    # For some layers, output features will be multi-dimensional tensors.
    # We flatten the feature tensors to vectors for easier storage in Spark DataFrames.
    output = [p.flatten() for p in preds]
    return pd.Series(output)

@pandas_udf('array<float>', PandasUDFType.SCALAR_ITER)
def featurize_udf(content_series_iter):
    '''
    This method is a Scalar Iterator pandas UDF wrapping our featurization function.
    The decorator specifies that this returns a Spark DataFrame column of type ArrayType(FloatType).

    :param content_series_iter: This argument is an iterator over batches of data, where each batch
                              is a pandas Series of image data.
    '''
    # With Scalar Iterator pandas UDFs, we can load the model once and then re-use it
    # for multiple data batches.  This amortizes the overhead of loading big models.
    model = model_fn()
    for content_series in content_series_iter:
        yield featurize_series(model, content_series)

PySparkImportError: [UNSUPPORTED_PACKAGE_VERSION] Pandas >= 2.2.0 must be installed; however, your version is 2.0.3.

In [28]:
!pip show Pandas | grep Version

Version: 2.3.0
    Licensed under the Apache License, Version 2.0 (the "License");
                            Version 2.0, January 2004
 Python Software Foundation License Version 2.
 the documentation are dual licensed under the PSF License Version 2
 Version 2.0, January 2004
    Licensed under the Apache License, Version 2.0 (the "License");


In [ ]:
features_df = images.repartition(20).select(col("path"),
                                            col("label"),
                                            featurize_udf("content").alias("features")
                                           )

In [ ]:
# spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1024")

In [ ]:
print(PATH_Result)

In [ ]:
features_df.write.mode("overwrite").parquet(PATH_Result)

In [ ]:
df = pd.read_parquet(PATH_Result, engine='pyarrow')

In [ ]:
df.head()

In [ ]:
df.loc[0,'features'].shape